<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.2-channel-flow/Ex09.2_00_geometry_lab.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.2 · Notebook 00 — geometry lab

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L9.2 · Turbulent Flow**

## What Ex_09.2 is about

Steady mean flow past an obstacle in a channel of length 4 and height 1, with
**your choice of shape** — circle, square, ellipse, diamond or aerofoil. A
physics-informed network solves the RANS momentum equations with a prescribed
eddy viscosity, incompressibility is built in through a stream function, and
the obstacle is described by a signed level set so that the solver never asks
which shape it was given. There is no exact solution: the set measures drag,
pressure drop and your own judgement, and the right answer may be *refusing to
answer* — saying precisely what would have to be true before a result could be
reported.

### Goals

By the end of the exercise set you can

1. write the steady RANS momentum residual with automatic differentiation, and
   say why there is no continuity term in the loss;
2. hard-enforce incompressibility through a **stream function**, and explain
   what that buys compared with penalising $\nabla\cdot\mathbf{u}$;
3. sample a domain that is **not a rectangle** — reject the obstacle, grade the
   points towards its surface, and say what the grading is for;
4. turn a trained field into the two numbers an engineer asks for, drag
   coefficient and pressure drop, and name what makes each of them unreliable;
5. compare shapes **at matched blockage**, and recognise an unmatched
   comparison as a measurement of area rather than of shape;
6. state the Reynolds number beyond which you would not report your own result,
   and what validation would be needed to change that.

### Method — five notebooks, run in order

| notebook | what you do | problem | lecture |
|---|---|---|---|
| **00** | nothing to write — shapes, level sets and sampling | the channel | — |
| **01** | write the residual and the loss | one obstacle in the channel | L9.2 |
| **02** | the interactive parameter study: geometry, Re, inlet speed, closure, sampling | one obstacle in the channel | L9.2 |
| **03** | five shapes compared at matched blockage | five obstacles | L9.2 |
| **04** | assemble the report for submission | the saved results | — |

Later notebooks load results saved in `Ex09.2_outputs/`. Everything runs on a
CPU; a single case takes one to three minutes, so budget 30–60 minutes for
notebooks 02 and 03 together.

### Applications

* **Drag of a body in a duct.** The drag coefficient of an obstacle, and how it
  changes with shape once the blockage ratio is held fixed.
* **Pressure drop across an obstruction.** The number that sizes a pump or a
  fan for a channel with an insert in it.
* **Knowing when not to report.** A prescribed eddy viscosity is the whole of
  this model's power and the whole of its weakness; the set ends with the
  Reynolds number beyond which the result should not be handed to anyone.

## What this notebook does

**Read and run; you are not asked to rewrite this.**

Steady mean flow through a channel containing one obstacle:

$$(\mathbf{u}\cdot\nabla)\mathbf{u} = -\nabla p
+ \nu_{\mathrm{eff}}\nabla^2\mathbf{u}, \qquad \nabla\cdot\mathbf{u} = 0$$

with $\nu_{\mathrm{eff}} = \nu + \nu_t$ — the effective viscosity of L9.2.
Prescribing $\nu_t$ turns an unsolvable turbulent problem into a laminar
one.

Incompressibility is hard-enforced by a stream function, so there is no
continuity loss term at all.

## The three modules

Every Part 2 exercise has the same three files beside it. The first two are
identical in every set; only the third changes.

| | |
|---|---|
| `course_core.py` | shared by the whole course — `set_seed`, `MLP`, `to_tensor`, `check` |
| `pinn_core.py` | the PDE machinery — `grad`, `d2`, samplers, `train_two_stage` |
| `problem.py` | **this** problem — shapes, level sets, the channel samplers, drag and pressure drop |

`problem.py` carries its own samplers, and that is the point of this notebook.
The shared library samples **rectangles**; this domain is a rectangle with a
hole in it, and the hole moves. A set with real geometry supplies the sampler
that knows about it.

One consequence to keep in mind everywhere below: the samplers return **NumPy
arrays**, not tensors. Anything the network is evaluated at gets wrapped with
`to_tensor(..., requires_grad=True)` at the point of use — and here that means
*every* point set, because the velocity is itself a derivative of the stream
function.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.2-channel-flow/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The five shapes

In [ ]:
print("shapes:", pb.SHAPES)

fig, ax = plt.subplots(1, 5, figsize=(14, 2.6))
for a, sh in zip(ax, pb.SHAPES):
    o = pb.Obstacle(sh, 1.2, 0.5, 0.2)
    xs, ys = o.outline()
    a.fill(xs, ys, alpha=0.6); a.plot(xs, ys, lw=1.2)
    a.set_aspect("equal"); a.set_title(f"{sh}\nD = {o.D:.2f}")
    a.set_xlim(0.85, 1.55); a.set_ylim(0.2, 0.8)
plt.tight_layout(); plt.show()

## 2 · The level set is what makes shape a parameter

Every shape exposes the same `phi(x, y)`: negative inside, zero on the surface,
positive outside. The solver never asks which shape it is — so adding a new one
means writing one function, not touching the physics.

In [ ]:
o = pb.Obstacle("aerofoil", 1.2, 0.5, 0.25, aspect=0.8)
g = np.linspace(0.5, 2.2, 260); h = np.linspace(0.05, 0.95, 130)
X, Y = np.meshgrid(g, h)
plt.figure(figsize=(9, 2.8))
cf = plt.contourf(X, Y, o.phi(X, Y), 40, cmap="coolwarm")
plt.contour(X, Y, o.phi(X, Y), [0.0], colors="k", linewidths=1.6)
plt.colorbar(cf, label="phi"); plt.gca().set_aspect("equal")
plt.title("level set: the black line is the surface"); plt.show()

# sanity checks the solver relies on
assert o.phi(np.array([1.2]), np.array([0.5]))[0] < 0     # inside
assert o.phi(np.array([3.5]), np.array([0.5]))[0] > 0     # far downstream
print("level set checks passed")

## 3 · Sampling, with grading towards the surface

`pb.sample_channel` draws candidate batches from the shared rectangle sampler
over the channel's bounding box and rejects everything inside the obstacle;
`pb.sample_obstacle` returns the surface and a graded cloud just outside it.
Both hand back NumPy, so the arrays below can be plotted directly.

In [ ]:
cfg = pb.PipeConfig(shape="circle", size=0.2, reynolds=100)
pb.plot_shape(cfg)
print(cfg, "\n nu_eff =", cfg.nu_eff, " blockage =", round(cfg.blockage, 3))

pts = pb.sample_channel(2000, cfg.obstacle, seed=1)
surf, near = pb.sample_obstacle(cfg.n_surface, cfg.obstacle, seed=1)
check_shape("interior points", pts, (2000, 2))
print(f"  smallest phi in the interior set : "
      f"{cfg.obstacle.phi(pts[:, 0], pts[:, 1]).min():.4f}   (must be > 0)")
print(f"  surface points                   : {len(surf)}")
print(f"  graded near-wall points          : {len(near)}")

**What you should see.** A cloud filling the channel with a clean hole where
the obstacle is, and a dense band hugging that hole. The smallest level-set
value in the interior set is positive by construction — `sample_channel` keeps
only points with `phi > 0.04`, so nothing sits on the surface itself.

That dense band is the graded near-wall cloud. It lies in the fluid, so
`run_case` adds it to the interior points and the momentum residual is enforced
there; no-slip is applied on the surface itself. Without the band the layer next
to the surface is barely sampled, and drag comes out far too low — *Where the
Points Must Go* in L9.1.

Next: **[notebook 01](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.2-channel-flow/Ex09.2_01_channel_flow.ipynb)**, where you write the residual and the loss.